# UTS Graf Sains Data — Soal pendahuluan 1.2 dan 1.3

> Jalankan Neo4j terlebih dahulu jika ingin menjalankan bagian driver/import. Untuk Neo4j lokal single-instance, gunakan `bolt://127.0.0.1:7687`.


## 0. Instalasi dan konfigurasi

In [18]:
%pip install -q neo4j pandas tqdm matplotlib networkx scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from pathlib import Path
import os
import zipfile
import shutil
import pandas as pd
import numpy as np
from neo4j import GraphDatabase
from tqdm.auto import tqdm

URI = os.getenv("NEO4J_URI", "bolt://127.0.0.1:7687")
USER = os.getenv("NEO4J_USER", "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD", "GSD12345")
DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
ZIP_PATH = Path("data.zip")
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 10_000

if not ZIP_PATH.exists():
    candidates = list(Path.cwd().glob("*.zip"))
    if candidates: ZIP_PATH = candidates[0]
if not ZIP_PATH.exists(): raise FileNotFoundError("data.zip tidak ditemukan di folder notebook.")

with zipfile.ZipFile(ZIP_PATH) as z:
    for member in ["data/events_p1.csv", "data/events_p2.csv"]:
        if member not in z.namelist(): raise FileNotFoundError(f"{member} tidak ada di ZIP")
        target=Path(member); target.parent.mkdir(parents=True,exist_ok=True)
        with z.open(member) as src, open(target,"wb") as dst: shutil.copyfileobj(src,dst)

P1_PATH=DATA_DIR/"events_p1.csv"
P2_PATH=DATA_DIR/"events_p2.csv"
print("URI:",URI); print("Username:",USER); print("Database:",DATABASE)
print("P1:",P1_PATH); print("P2:",P2_PATH)


URI: bolt://127.0.0.1:7687
Username: neo4j
Database: neo4j
P1: data\events_p1.csv
P2: data\events_p2.csv


## 1.2 — Mempelajari penggunaan driver Neo4j

### Membuat node dan relationship
Query `MERGE` membuat node `Person` jika belum ada, lalu membuat relationship berarah `KNOWS`. Fungsi baca menggunakan `RoutingControl.READ` dan mengurutkan nama teman.

In [20]:
from neo4j import GraphDatabase

def add_friend(driver, name, friend_name):
    driver.execute_query(
        "MERGE (a:Person {name: $name}) MERGE (friend:Person {name: $friend_name}) MERGE (a)-[:KNOWS]->(friend)",
        name=name, friend_name=friend_name, database_=DATABASE,
    )

def print_friends(driver, name):
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name RETURN friend.name AS friend_name ORDER BY friend.name",
        name=name, database_=DATABASE,
    )
    for record in records: print(record["friend_name"])

with GraphDatabase.driver(URI, auth=(USER, PASSWORD)) as driver:
    driver.verify_connectivity()
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    print("Teman Arthur:"); print_friends(driver, "Arthur")


Teman Arthur:
Guinevere
Lancelot
Merlin


### Jawaban checkpoint 1.2

1. **Nama database:** `neo4j`, terlihat dari argumen `database_="neo4j"` pada contoh tutorial.
2. **Akses lain dalam satu instance:** Cypher untuk membaca (`MATCH`, `RETURN`), membuat (`CREATE`), menggabungkan/menghindari duplikasi (`MERGE`), memperbarui (`SET`), menghapus (`DELETE`/`DETACH DELETE`), membuat constraint/index (`CREATE CONSTRAINT`, `CREATE INDEX`), serta memilih database lewat session/parameter `database`. Hak akses tetap bergantung pada konfigurasi dan privilege pengguna.
3. **Jenis graf pertemanan:** graf berarah (directed), karena relationship dibuat sebagai `(a)-[:KNOWS]->(friend)`. Tidak otomatis ada edge balik dari teman ke Arthur; graf juga dapat memiliki banyak node dan edge.

### Opsional: reset database

**Jangan jalankan kecuali database ini khusus latihan dan Anda memang ingin menghapus seluruh node serta relationship.**

In [21]:
# PERINGATAN: destruktif untuk seluruh isi database yang dipilih.
# Uncomment hanya jika benar-benar diperlukan:
# with GraphDatabase.driver(URI, auth=(USER, PASSWORD)) as driver:
#     driver.execute_query("MATCH (n) DETACH DELETE n", database_=DATABASE)

## 1.3 — Impor data RetailRocket

Data upload menyediakan dua periode: `events_p1.csv` (15–28 Mei 2015) dan `events_p2.csv` (29 Mei–4 Juni 2015). Keduanya diimpor ke Neo4j. `events.csv` tidak digunakan karena isinya hanya pointer Git LFS.


In [22]:
# Data event tersedia sebagai dua file periode di ZIP.
CSV_PATHS = [P1_PATH, P2_PATH]
for path in CSV_PATHS:
    print("CSV:", path, "| ukuran MB:", round(path.stat().st_size/1e6, 2))
    print(pd.read_csv(path, nrows=5))


CSV: data\events_p1.csv | ukuran MB: 10.41
       timestamp  visitorid event  itemid  transactionid
0  1431660377681     310356  view  454137            NaN
1  1431658821719      73735  view  138284            NaN
2  1431658259160    1294857  view  249086            NaN
3  1431660749956    1244172  view  299710            NaN
4  1431658621087     944537  view  410870            NaN
CSV: data\events_p2.csv | ukuran MB: 4.97
       timestamp  visitorid event  itemid  transactionid
0  1433221332117     257597  view  355908            NaN
1  1433224214164     992329  view  248676            NaN
2  1433221999827     111016  view  318965            NaN
3  1433221955914     483717  view  253185            NaN
4  1433221337106     951259  view  367447            NaN


In [23]:
def create_constraints(session):
    session.run("CREATE CONSTRAINT visitor_id IF NOT EXISTS FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE").consume()
    session.run("CREATE CONSTRAINT item_id IF NOT EXISTS FOR (i:Item) REQUIRE i.itemId IS UNIQUE").consume()

def import_batch(tx, batch):
    query = """
    UNWIND $batch AS row

    MERGE (v:Visitor {visitorId: toInteger(row.visitorid)})
    MERGE (i:Item {itemId: toInteger(row.itemid)})

    FOREACH (_ IN CASE
        WHEN row.event = 'view' THEN [1]
        ELSE []
    END |
        CREATE (v)-[:VIEWED {
            timestamp: toInteger(row.timestamp)
        }]->(i)
    )

    FOREACH (_ IN CASE
        WHEN row.event = 'addtocart' THEN [1]
        ELSE []
    END |
        CREATE (v)-[:ADDED_TO_CART {
            timestamp: toInteger(row.timestamp)
        }]->(i)
    )

    FOREACH (_ IN CASE
        WHEN row.event = 'transaction' THEN [1]
        ELSE []
    END |
        CREATE (v)-[:BOUGHT {
            timestamp: toInteger(row.timestamp),
            transactionId: row.transactionid
        }]->(i)
    )

    RETURN count(*) AS processed
    """

    tx.run(query, batch=batch).consume()

total_rows = sum(len(chunk) for path in CSV_PATHS for chunk in pd.read_csv(path,usecols=["timestamp"],chunksize=250_000))
print("Jumlah event P1+P2:",f"{total_rows:,}")


Jumlah event P1+P2: 450,039


In [ ]:
import math

with GraphDatabase.driver(
    URI,
    auth=(USER, PASSWORD)
) as driver:

    driver.verify_connectivity()

    with driver.session(database=DATABASE) as session:

        create_constraints(session)

        for path in CSV_PATHS:

            print(f"\nMemproses: {path}")

            total_rows = len(
                pd.read_csv(
                    path,
                    usecols=["timestamp"]
                )
            )

            total_batches = math.ceil(
                total_rows / BATCH_SIZE
            )

            chunks = pd.read_csv(
                path,
                chunksize=BATCH_SIZE
            )

            for chunk in tqdm(
                chunks,
                total=total_batches,
                desc=f"Import {path.name}"
            ):

                if "transactionid" in chunk.columns:
                    chunk["transactionid"] = chunk[
                        "transactionid"
                    ].where(
                        chunk["transactionid"].notna(),
                        None
                    )

                chunk["visitorid"] = pd.to_numeric(
                    chunk["visitorid"],
                    errors="coerce"
                )

                chunk["itemid"] = pd.to_numeric(
                    chunk["itemid"],
                    errors="coerce"
                )

                chunk["timestamp"] = pd.to_numeric(
                    chunk["timestamp"],
                    errors="coerce"
                )

                chunk = chunk.dropna(
                    subset=[
                        "visitorid",
                        "itemid",
                        "timestamp",
                        "event"
                    ]
                )

                records = chunk.to_dict(
                    orient="records"
                )

                if records:
                    session.execute_write(
                        import_batch,
                        records
                    )


Memproses: data\events_p1.csv


Import events_p1.csv: 0it [00:00, ?it/s]


Memproses: data\events_p2.csv


Import events_p2.csv: 0it [00:00, ?it/s]


IMPORT P1 + P2 SELESAI


### Validasi hasil impor dengan Cypher

Query berikut menghitung jumlah node menurut label dan relationship menurut tipe.

In [25]:
with GraphDatabase.driver(URI, auth=(USER, PASSWORD)) as driver:
    with driver.session(database=DATABASE) as session:
        print("Jumlah node:")
        for r in session.run("MATCH (n) RETURN labels(n) AS labels, count(*) AS jumlah ORDER BY labels"):
            print(r["labels"], r["jumlah"])
        print("Jumlah relationship:")
        for r in session.run("MATCH ()-[r]->() RETURN type(r) AS tipe, count(*) AS jumlah ORDER BY tipe"):
            print(r["tipe"], r["jumlah"])

Jumlah node:
['Item'] 235061
['Person'] 4
['User'] 140366
['Visitor'] 1407580
Jumlah relationship:
ADDED_TO_CART 108779
BOUGHT 35759
KNOWS 3
SIMILAR_TO 10
VIEWED 4143478
